# ЛР-02: Рационы и медкомплекты

## Worked example: military 01

Это полностью разобранный example. Он показывает образец постановки, решения и интерпретации транспортной модели.

## 1. Постановка кейса

Военный закрытый кейс с нормальной балансировкой запасов и потребностей.

### Запасы

| Поставщик | Объём |
| --- | --- |
| База Север | 24 |
| База Центр | 30 |
| База Юг | 26 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Группа 1 | 18 |
| Группа 2 | 20 |
| Группа 3 | 16 |
| Группа 4 | 26 |

### Матрица затрат

| Откуда / Куда | Группа 1 | Группа 2 | Группа 3 | Группа 4 |
| --- | --- | --- | --- | --- |
| База Север | 5 | 6 | 8 | 9 |
| База Центр | 4 | 5 | 7 | 8 |
| База Юг | 7 | 5 | 4 | 6 |

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog


DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"
DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
BALANCE_TOLERANCE = 1e-9


def make_vector_frame(values, labels, value_name):
    """Formats a one-dimensional vector as a readable table.

    Args:
        values (np.ndarray): Numeric vector to display.
        labels (list[str]): Row labels for the vector elements.
        value_name (str): Name of the numeric column.

    Returns:
        pd.DataFrame: Table with labels and one numeric column.
    """

    return pd.DataFrame({value_name: values}, index=labels)


def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Balances an open transport task by adding one dummy node if needed.

    Args:
        supplies (np.ndarray): Supply vector ``a_i``.
        demands (np.ndarray): Demand vector ``b_j``.
        costs (np.ndarray): Cost matrix ``c_ij``.
        supplier_names (list[str]): Labels for supply rows.
        consumer_names (list[str]): Labels for demand columns.
        dummy_cost (float): Neutral cost for a dummy row or column.

    Returns:
        tuple: Balanced supplies, demands, costs, labels, and a short note.
    """

    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    if balance_difference > BALANCE_TOLERANCE:
        balanced_demands = np.append(balanced_demands, balance_difference)
        balanced_consumer_names.append(DUMMY_CONSUMER_NAME)
        dummy_column = np.full(len(balanced_supplies), dummy_cost)
        balanced_costs = np.column_stack([balanced_costs, dummy_column])
        balance_note = "Добавлен фиктивный потребитель для избыточного запаса."
    elif balance_difference < -BALANCE_TOLERANCE:
        shortage = -balance_difference
        balanced_supplies = np.append(balanced_supplies, shortage)
        balanced_supplier_names.append(DUMMY_SUPPLIER_NAME)
        dummy_row = np.full(len(balanced_demands), dummy_cost)
        balanced_costs = np.vstack([balanced_costs, dummy_row])
        balance_note = "Добавлен фиктивный поставщик для дефицита."
    else:
        balance_note = "Задача уже закрытая: суммарный запас равен спросу."

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )


def make_route_labels(supplier_count, consumer_count):
    """Creates compact labels for flattened variables ``x_ij``.

    Args:
        supplier_count (int): Number of rows in the transport matrix.
        consumer_count (int): Number of columns in the transport matrix.

    Returns:
        list[str]: Labels ordered exactly like ``costs.flatten()``.
    """

    route_labels = []
    for supplier_idx in range(supplier_count):
        for consumer_idx in range(consumer_count):
            route_labels.append(f"x_{supplier_idx + 1},{consumer_idx + 1}")

    return route_labels


def make_constraint_labels(supplier_names, consumer_names):
    """Creates labels for rows of ``A_eq`` and values of ``b_eq``.

    Args:
        supplier_names (list[str]): Labels for supply constraints.
        consumer_names (list[str]): Labels for demand constraints.

    Returns:
        list[str]: Constraint labels in the same order as ``b_eq``.
    """

    supplier_constraints = [f"запас: {name}" for name in supplier_names]
    consumer_constraints = [f"спрос: {name}" for name in consumer_names]

    return supplier_constraints + consumer_constraints


def build_transport_lp(supplies, demands, costs):
    """Builds the canonical LP arrays for ``scipy.optimize.linprog``.

    Args:
        supplies (np.ndarray): Balanced supply vector ``a_i``.
        demands (np.ndarray): Balanced demand vector ``b_j``.
        costs (np.ndarray): Balanced cost matrix ``c_ij``.

    Returns:
        dict[str, object]: Model parts ``c``, ``A_eq``, ``b_eq``, and ``bounds``.
    """

    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    c = costs.flatten()
    A_eq_rows = []
    b_eq_values = []

    # Step 1: add supply constraints for every row of the transport table.
    for supplier_idx in range(supplier_count):
        row = np.zeros(variable_count)
        for consumer_idx in range(consumer_count):
            route_index = supplier_idx * consumer_count + consumer_idx
            row[route_index] = 1.0
        A_eq_rows.append(row)
        b_eq_values.append(supplies[supplier_idx])

    # Step 2: add demand constraints for every column of the transport table.
    for consumer_idx in range(consumer_count):
        row = np.zeros(variable_count)
        for supplier_idx in range(supplier_count):
            route_index = supplier_idx * consumer_count + consumer_idx
            row[route_index] = 1.0
        A_eq_rows.append(row)
        b_eq_values.append(demands[consumer_idx])

    A_eq = np.array(A_eq_rows)
    b_eq = np.array(b_eq_values)
    bounds = [(0.0, None)] * variable_count

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
    }


def solve_transport_problem(supplies, demands, costs):
    """Solves a balanced transport problem with ``linprog``.

    Args:
        supplies (np.ndarray): Balanced supply vector.
        demands (np.ndarray): Balanced demand vector.
        costs (np.ndarray): Balanced cost matrix.

    Returns:
        tuple: ``OptimizeResult``, plan matrix, and LP model dictionary.

    Raises:
        RuntimeError: If HiGHS cannot solve the LP model.
    """

    lp_model = build_transport_lp(supplies, demands, costs)
    supplier_count, consumer_count = costs.shape

    result = linprog(
        lp_model["c"],
        A_eq=lp_model["A_eq"],
        b_eq=lp_model["b_eq"],
        bounds=lp_model["bounds"],
        method="highs",
    )

    if not result.success:
        raise RuntimeError(result.message)

    plan = result.x.reshape(supplier_count, consumer_count)

    return result, plan, lp_model


def make_plan_frame(plan, supplier_names, consumer_names):
    """Formats the optimized shipment matrix as a table.

    Args:
        plan (np.ndarray): Optimized shipment matrix ``X``.
        supplier_names (list[str]): Row labels.
        consumer_names (list[str]): Column labels.

    Returns:
        pd.DataFrame: Readable shipment plan.
    """

    return pd.DataFrame(plan, index=supplier_names, columns=consumer_names)


def make_used_routes_frame(plan_df, cost_df, tolerance=BALANCE_TOLERANCE):
    """Lists all non-zero routes in the optimized plan.

    Args:
        plan_df (pd.DataFrame): Shipment plan table.
        cost_df (pd.DataFrame): Cost matrix table with the same shape.
        tolerance (float): Small numerical threshold for non-zero routes.

    Returns:
        pd.DataFrame: Table of active routes with volumes and costs.
    """

    used_routes = []

    for supplier_name in plan_df.index:
        for consumer_name in plan_df.columns:
            volume = float(plan_df.loc[supplier_name, consumer_name])
            if volume <= tolerance:
                continue

            unit_cost = float(cost_df.loc[supplier_name, consumer_name])
            is_dummy_route = (
                supplier_name == DUMMY_SUPPLIER_NAME
                or consumer_name == DUMMY_CONSUMER_NAME
            )
            route_type = "фиктивный" if is_dummy_route else "реальный"

            used_routes.append(
                {
                    "маршрут": f"{supplier_name} -> {consumer_name}",
                    "тип": route_type,
                    "объем": round(volume, 2),
                    "тариф": unit_cost,
                    "затраты": round(volume * unit_cost, 2),
                }
            )

    return pd.DataFrame(used_routes)


def make_balance_check_frames(plan_df, supplies, demands):
    """Builds row and column balance check tables.

    Args:
        plan_df (pd.DataFrame): Shipment plan table.
        supplies (np.ndarray): Balanced supply vector.
        demands (np.ndarray): Balanced demand vector.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Supply and demand check tables.
    """

    supply_check_df = pd.DataFrame(
        {
            "план": plan_df.sum(axis=1),
            "запас": supplies,
            "разница": plan_df.sum(axis=1) - supplies,
        },
        index=plan_df.index,
    )

    demand_check_df = pd.DataFrame(
        {
            "план": plan_df.sum(axis=0),
            "спрос": demands,
            "разница": plan_df.sum(axis=0) - demands,
        },
        index=plan_df.columns,
    )

    return supply_check_df, demand_check_df


In [2]:
# Step 1: write the case data exactly as it appears in the task statement.
supplier_names = [
    'База Север',
    'База Центр',
    'База Юг',
]
consumer_names = [
    'Группа 1',
    'Группа 2',
    'Группа 3',
    'Группа 4',
]

supplies = np.array(
[
    24,
    30,
    26,
],
dtype=float,
)

demands = np.array(
[
    18,
    20,
    16,
    26,
],
dtype=float,
)

costs = np.array(
[
    [5, 6, 8, 9],
    [4, 5, 7, 8],
    [7, 5, 4, 6],
],
dtype=float,
)

# Step 2: show the input vectors and the cost matrix as readable tables.
supply_df = make_vector_frame(supplies, supplier_names, "запас")
demand_df = make_vector_frame(demands, consumer_names, "спрос")
cost_df_raw = pd.DataFrame(costs, index=supplier_names, columns=consumer_names)

print("Запасы поставщиков a_i:")
display(supply_df)

print("Спрос потребителей b_j:")
display(demand_df)

print("Матрица затрат c_ij:")
display(cost_df_raw)

# Step 3: check and, if needed, close the transport model.
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print("sum supply =", balanced_supplies.sum())
print("sum demand =", balanced_demands.sum())

assert np.allclose(
    balanced_supplies.sum(),
    balanced_demands.sum(),
), "После балансировки сумма запасов должна равняться сумме спроса."

# Step 4: solve the canonical LP model min c^T x, A_eq x = b_eq, x >= 0.
result, plan, lp_model = solve_transport_problem(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
)

route_labels = make_route_labels(
    len(balanced_supplier_names),
    len(balanced_consumer_names),
)
constraint_labels = make_constraint_labels(
    balanced_supplier_names,
    balanced_consumer_names,
)

c_df = pd.DataFrame(
    {"переменная": route_labels, "стоимость c": lp_model["c"]}
)
A_eq_df = pd.DataFrame(lp_model["A_eq"], columns=route_labels)
b_eq_df = pd.DataFrame(
    {"ограничение": constraint_labels, "b_eq": lp_model["b_eq"]}
)

print("Вектор цели c = costs.flatten():")
display(c_df)

print("Матрица ограничений A_eq:")
display(A_eq_df)

print("Вектор правых частей b_eq:")
display(b_eq_df)

plan_df = make_plan_frame(
    plan,
    balanced_supplier_names,
    balanced_consumer_names,
)
cost_df = pd.DataFrame(
    balanced_costs,
    index=balanced_supplier_names,
    columns=balanced_consumer_names,
)

print("Оптимальная стоимость:", round(result.fun, 2))
print("План перевозок X:")
display(plan_df)

print("Матрица затрат после балансировки:")
display(cost_df)


Запасы поставщиков a_i:


,запас
База Север,24.0
База Центр,30.0
База Юг,26.0


Спрос потребителей b_j:


,спрос
Группа 1,18.0
Группа 2,20.0
Группа 3,16.0
Группа 4,26.0


Матрица затрат c_ij:


...

,ограничение,b_eq
0,запас: База Север,24.0
1,запас: База Центр,30.0
2,запас: База Юг,26.0
3,спрос: Группа 1,18.0
4,спрос: Группа 2,20.0
5,спрос: Группа 3,16.0
6,спрос: Группа 4,26.0


Оптимальная стоимость: 448.0
План перевозок X:


,Группа 1,Группа 2,Группа 3,Группа 4
База Север,0.0,20.0,0.0,4.0
База Центр,18.0,0.0,0.0,12.0
База Юг,0.0,0.0,16.0,10.0


Матрица затрат после балансировки:


,Группа 1,Группа 2,Группа 3,Группа 4
База Север,5.0,6.0,8.0,9.0
База Центр,4.0,5.0,7.0,8.0
База Юг,7.0,5.0,4.0,6.0


In [3]:
# Step 5: list active routes and check feasibility of the optimal plan.
used_routes_df = make_used_routes_frame(plan_df, cost_df)

supply_check_df, demand_check_df = make_balance_check_frames(
    plan_df,
    balanced_supplies,
    balanced_demands,
)

print("Использованные маршруты:")
display(used_routes_df)

print("Проверка баланса по поставщикам:")
display(supply_check_df)

print("Проверка баланса по потребителям:")
display(demand_check_df)

assert np.allclose(
    supply_check_df["план"],
    supply_check_df["запас"],
), "Суммы по строкам должны совпадать с запасами."

assert np.allclose(
    demand_check_df["план"],
    demand_check_df["спрос"],
), "Суммы по столбцам должны совпадать со спросом."

assert np.allclose(
    float((plan_df * cost_df).to_numpy().sum()),
    result.fun,
), "Стоимость по таблицам должна совпадать с result.fun."


Использованные маршруты:


,маршрут,тип,объем,тариф,затраты
0,База Север -> Группа 2,реальный,20.0,6.0,120.0
1,База Север -> Группа 4,реальный,4.0,9.0,36.0
2,База Центр -> Группа 1,реальный,18.0,4.0,72.0
3,База Центр -> Группа 4,реальный,12.0,8.0,96.0
4,База Юг -> Группа 3,реальный,16.0,4.0,64.0
5,База Юг -> Группа 4,реальный,10.0,6.0,60.0


Проверка баланса по поставщикам:


,план,запас,разница
База Север,24.0,24.0,0.0
База Центр,30.0,30.0,0.0
База Юг,26.0,26.0,0.0


Проверка баланса по потребителям:


,план,спрос,разница
Группа 1,18.0,18.0,0.0
Группа 2,20.0,20.0,0.0
Группа 3,16.0,16.0,0.0
Группа 4,26.0,26.0,0.0


## 2. Что важно проговорить в выводе

- сначала проверьте, закрытая задача или открытая;
- после балансировки объясните смысл фиктивного узла, если он появился;
- в LP-форме явно покажите `c`, `A_eq`, `b_eq`, `bounds`;
- после решения сверяйте суммы по строкам и столбцам;
- отдельно перечисляйте ненулевые реальные маршруты;
- фиктивные перевозки не являются реальной доставкой: это резерв или дефицит.

Для отчёта недостаточно назвать только оптимальную стоимость. Важно показать,
почему план допустим и как его читать на языке предметной области.
